In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage


d:\Lakshya\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()
model =  ChatOllama(model = 'minimax-m3:cloud', temperature= 0.5)

# define the state
class ChatState(TypedDict):
    # define the properties
    messages : Annotated[list[BaseMessage], add_messages]

# define the methord
def chat_node(State: ChatState):
    messages = State['messages']

    # define the LLM
    response =  model.invoke(messages)
    return {'messages': [response]}

In [4]:
graph = StateGraph(ChatState)
# add node
graph.add_node('chat_node', chat_node)

# add edge
graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

Checkpointer = InMemorySaver()
chatbot = graph.compile(checkpointer= Checkpointer)
config = {"configurable":{"thread_id":"1"}}
result = chatbot.stream({"messages":[HumanMessage(content = "Explain bout langgraph in detail and give the demo code about graph")]}, config= config , stream_mode= 'messages')
print(result)

<generator object Pregel.stream at 0x0000025ECF95F870>


In [5]:
for chunk, metadata in result:
    if hasattr(chunk, "content"):
        print(chunk.content, end="", flush=True)

# LangGraph: A Comprehensive Guide

## What is LangGraph?

**LangGraph** is a library built by the LangChain team for building **stateful, multi-actor applications** with Large Language Models (LLMs). It extends LangChain by allowing you to orchestrate complex workflows as **graphs** (rather than simple chains), making it ideal for building:

- 🤖 **AI Agents** with cyclic decision-making
- 💬 **Multi-turn conversational systems**
- 🔄 **Workflows with loops and conditional logic**
- 👥 **Multi-agent systems** (supervisor/worker patterns)
- 🧠 **Complex RAG pipelines** with self-correction

---

## Core Concepts

### 1. **State**
A shared data structure (typically a `TypedDict` or `Pydantic` model) that flows through the graph. Every node reads from and updates this state.

```python
from typing import TypedDict

class AgentState(TypedDict):
    messages: list
    next_step: str
```

### 2. **Nodes**
Python functions that perform actions. Each node:
- Takes the current state as input
- Retu